# Ireflix Big Data Analytics - CA1
**Module:** Big Data Analytical Methods  
**Programme:** L7 Diploma in Data Analytics  
**Submission Deadline:** Sunday 31st May 2026

---
## Part 1: Database Design & Implementation (60 Marks)
### Step 1: Install dependencies and create the SQLite database

In [ ]:
import sqlite3
import random
import string
from faker import Faker
import pandas as pd

fake = Faker()
random.seed(42)
Faker.seed(42)

# Connect to SQLite database (creates file if not exists)
conn = sqlite3.connect('ireflix.db')
cursor = conn.cursor()
print('Database connection established: ireflix.db')

### Step 2: Create Tables with PKs, FKs, Constraints

In [ ]:
# Drop tables if they exist (for re-runs)
cursor.executescript('''
    PRAGMA foreign_keys = ON;
    DROP TABLE IF EXISTS UserMovieActivity;
    DROP TABLE IF EXISTS UserSeriesActivity;
    DROP TABLE IF EXISTS Movies;
    DROP TABLE IF EXISTS Series;
    DROP TABLE IF EXISTS Actors;
    DROP TABLE IF EXISTS Directors;
    DROP TABLE IF EXISTS Users;
''')

# ── Users ──────────────────────────────────────────────────────────────────
cursor.execute('''
    CREATE TABLE Users (
        userName   VARCHAR(35) PRIMARY KEY,
        firstName  VARCHAR(35) NOT NULL,
        surName    VARCHAR(35) NOT NULL,
        password   VARCHAR(15) NOT NULL
    )
''')

# ── Actors ─────────────────────────────────────────────────────────────────
# Assumption: no two actors share the same surname (per spec)
cursor.execute('''
    CREATE TABLE Actors (
        actorFirstName  VARCHAR(35) NOT NULL,
        surName         VARCHAR(35) PRIMARY KEY,
        nationality     VARCHAR(35)
    )
''')

# ── Directors ──────────────────────────────────────────────────────────────
cursor.execute('''
    CREATE TABLE Directors (
        nameFirst   VARCHAR(35) NOT NULL,
        surName     VARCHAR(35) PRIMARY KEY,
        nationality VARCHAR(35)
    )
''')

# ── Movies ─────────────────────────────────────────────────────────────────
cursor.execute('''
    CREATE TABLE Movies (
        movieID          CHAR(6)     PRIMARY KEY
                             CHECK(movieID GLOB '[0-9][0-9][0-9][0-9][0-9][0-9]'),
        movieName        VARCHAR(35) NOT NULL,
        genre            VARCHAR(20),
        duration_minutes INTEGER     CHECK(duration_minutes BETWEEN 1 AND 180),
        rating           INTEGER     CHECK(rating BETWEEN 1 AND 5),
        actorFirstName   VARCHAR(35),
        actorSurName     VARCHAR(35),
        directorSurName  VARCHAR(35),
        release_year     INTEGER,
        FOREIGN KEY (actorSurName)    REFERENCES Actors(surName),
        FOREIGN KEY (directorSurName) REFERENCES Directors(surName)
    )
''')

# ── Series ─────────────────────────────────────────────────────────────────
cursor.execute('''
    CREATE TABLE Series (
        seriesID         CHAR(6)     PRIMARY KEY
                             CHECK(seriesID GLOB '[0-9][0-9][0-9][0-9][0-9][0-9]'),
        seriesName       VARCHAR(35) NOT NULL,
        genre            VARCHAR(20),
        rating           INTEGER     CHECK(rating BETWEEN 1 AND 5),
        actorFirstName   VARCHAR(35),
        actorSurName     VARCHAR(35),
        directorSurName  VARCHAR(35),
        number_of_seasons INTEGER    CHECK(number_of_seasons >= 1),
        release_year     INTEGER,
        FOREIGN KEY (actorSurName)    REFERENCES Actors(surName),
        FOREIGN KEY (directorSurName) REFERENCES Directors(surName)
    )
''')

# ── Activity tables ────────────────────────────────────────────────────────
cursor.execute('''
    CREATE TABLE UserMovieActivity (
        activityID  INTEGER PRIMARY KEY AUTOINCREMENT,
        userName    VARCHAR(35) NOT NULL,
        movieID     CHAR(6)     NOT NULL,
        genre       VARCHAR(20),
        watched_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (userName) REFERENCES Users(userName),
        FOREIGN KEY (movieID)  REFERENCES Movies(movieID)
    )
''')

cursor.execute('''
    CREATE TABLE UserSeriesActivity (
        activityID  INTEGER PRIMARY KEY AUTOINCREMENT,
        userName    VARCHAR(35) NOT NULL,
        seriesID    CHAR(6)     NOT NULL,
        genre       VARCHAR(20),
        watched_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (userName)  REFERENCES Users(userName),
        FOREIGN KEY (seriesID)  REFERENCES Series(seriesID)
    )
''')

conn.commit()
print('All tables created successfully with PKs, FKs, and constraints.')

### Step 3: Generate Synthetic Data (1000+ records)

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────
GENRES = ['Action', 'Comedy', 'Drama', 'Horror', 'Romance',
          'Sci-Fi', 'Thriller', 'Documentary', 'Animation', 'Fantasy']

NATIONALITIES = ['American', 'British', 'Irish', 'French', 'German',
                 'Spanish', 'Italian', 'Australian', 'Canadian', 'Japanese']

def rand_id(length=6):
    return ''.join(random.choices(string.digits, k=length))

def rand_password(length=12):
    chars = string.ascii_letters + string.digits + '!@#$'
    return ''.join(random.choices(chars, k=length))

def trunc(s, n):
    return str(s)[:n]

# ── 1. Users (200) ─────────────────────────────────────────────────────────
users = set()
user_rows = []
while len(user_rows) < 200:
    uname = trunc(fake.user_name(), 35)
    if uname in users:
        continue
    users.add(uname)
    user_rows.append((
        uname,
        trunc(fake.first_name(), 35),
        trunc(fake.last_name(), 35),
        rand_password()
    ))
cursor.executemany('INSERT INTO Users VALUES (?,?,?,?)', user_rows)

# ── 2. Actors (150, unique surnames) ──────────────────────────────────────
actor_surnames = set()
actor_rows = []
# Seed with a well-known actor to use later in queries
KNOWN_ACTOR = ('Leonardo', 'DiCaprio')
actor_surnames.add('DiCaprio')
actor_rows.append(('Leonardo', 'DiCaprio', 'American'))

while len(actor_rows) < 150:
    sn = trunc(fake.last_name(), 35)
    if sn in actor_surnames:
        continue
    actor_surnames.add(sn)
    actor_rows.append((
        trunc(fake.first_name(), 35),
        sn,
        random.choice(NATIONALITIES)
    ))
cursor.executemany('INSERT INTO Actors VALUES (?,?,?)', actor_rows)

# ── 3. Directors (100, unique surnames) ───────────────────────────────────
dir_surnames = set(actor_surnames)  # directors can't share surname with actors either per spec
dir_rows = []
while len(dir_rows) < 100:
    sn = trunc(fake.last_name(), 35)
    if sn in dir_surnames:
        continue
    dir_surnames.add(sn)
    dir_rows.append((
        trunc(fake.first_name(), 35),
        sn,
        random.choice(NATIONALITIES)
    ))
cursor.executemany('INSERT INTO Directors VALUES (?,?,?)', dir_rows)

actor_sn_list  = [r[1] for r in actor_rows]
actor_fn_map   = {r[1]: r[0] for r in actor_rows}
dir_sn_list    = [r[1] for r in dir_rows]

# ── 4. Movies (600) ────────────────────────────────────────────────────────
movie_ids = set()
movie_rows = []
# Force DiCaprio into at least 10 movies
for _ in range(10):
    mid = rand_id()
    while mid in movie_ids:
        mid = rand_id()
    movie_ids.add(mid)
    g = random.choice(GENRES)
    movie_rows.append((
        mid,
        trunc(fake.catch_phrase(), 35),
        g,
        random.randint(60, 180),
        random.randint(1, 5),
        'Leonardo',
        'DiCaprio',
        random.choice(dir_sn_list),
        random.randint(1990, 2025)
    ))

while len(movie_rows) < 600:
    mid = rand_id()
    while mid in movie_ids:
        mid = rand_id()
    movie_ids.add(mid)
    asn = random.choice(actor_sn_list)
    g = random.choice(GENRES)
    movie_rows.append((
        mid,
        trunc(fake.catch_phrase(), 35),
        g,
        random.randint(60, 180),
        random.randint(1, 5),
        actor_fn_map[asn],
        asn,
        random.choice(dir_sn_list),
        random.randint(1990, 2025)
    ))

cursor.executemany(
    'INSERT INTO Movies VALUES (?,?,?,?,?,?,?,?,?)',
    movie_rows
)
movie_id_list = [r[0] for r in movie_rows]
movie_genre_map = {r[0]: r[2] for r in movie_rows}

# ── 5. Series (400) ────────────────────────────────────────────────────────
# Exclude DiCaprio from series actor pool (used in exclusion query later)
series_actor_pool = [sn for sn in actor_sn_list if sn != 'DiCaprio']
series_ids = set()
series_rows = []
while len(series_rows) < 400:
    sid = rand_id()
    while sid in series_ids or sid in movie_ids:
        sid = rand_id()
    series_ids.add(sid)
    asn = random.choice(series_actor_pool)
    g = random.choice(GENRES)
    series_rows.append((
        sid,
        trunc(fake.catch_phrase(), 35),
        g,
        random.randint(1, 5),
        actor_fn_map[asn],
        asn,
        random.choice(dir_sn_list),
        random.randint(1, 10),
        random.randint(1990, 2025)
    ))
cursor.executemany(
    'INSERT INTO Series VALUES (?,?,?,?,?,?,?,?,?)',
    series_rows
)
series_id_list = [r[0] for r in series_rows]
series_genre_map = {r[0]: r[2] for r in series_rows}

# ── 6. UserMovieActivity (500) ─────────────────────────────────────────────
uma_rows = []
user_list = [r[0] for r in user_rows]
for _ in range(500):
    mid = random.choice(movie_id_list)
    uma_rows.append((
        random.choice(user_list),
        mid,
        movie_genre_map[mid]
    ))
cursor.executemany(
    'INSERT INTO UserMovieActivity (userName,movieID,genre) VALUES (?,?,?)',
    uma_rows
)

# ── 7. UserSeriesActivity (500) ────────────────────────────────────────────
usa_rows = []
for _ in range(500):
    sid = random.choice(series_id_list)
    usa_rows.append((
        random.choice(user_list),
        sid,
        series_genre_map[sid]
    ))
cursor.executemany(
    'INSERT INTO UserSeriesActivity (userName,seriesID,genre) VALUES (?,?,?)',
    usa_rows
)

conn.commit()
print('Synthetic data inserted successfully.')
print(f'Users: 200 | Actors: 150 | Directors: 100')
print(f'Movies: 600 | Series: 400')
print(f'Movie activity: 500 | Series activity: 500')
print(f'Total records across all tables: {200+150+100+600+400+500+500}')

### Step 4: Verify Record Counts

In [ ]:
tables = ['Users','Actors','Directors','Movies','Series','UserMovieActivity','UserSeriesActivity']
for t in tables:
    cursor.execute(f'SELECT COUNT(*) FROM {t}')
    print(f'{t}: {cursor.fetchone()[0]} records')

### Step 5: PK/FK Relationship Verification

In [ ]:
# Show schema for each table
for t in tables:
    print(f'\n=== {t} ===')
    cursor.execute(f'PRAGMA table_info({t})')
    df = pd.DataFrame(cursor.fetchall(), columns=['cid','name','type','notnull','dflt','pk'])
    print(df[['name','type','notnull','pk']].to_string(index=False))

### Step 6: SQL Queries
#### Q6a – How many movies did Leonardo DiCaprio appear in?

In [ ]:
cursor.execute('''
    SELECT actorFirstName, actorSurName, COUNT(*) AS movies_count
    FROM Movies
    WHERE actorSurName = 'DiCaprio'
    GROUP BY actorFirstName, actorSurName
''')
result = cursor.fetchall()
print('DiCaprio movie count:', result)

#### Q6b – Actors who were protagonists in Movies but NOT in Series

In [ ]:
cursor.execute('''
    SELECT DISTINCT actorFirstName, actorSurName
    FROM Movies
    WHERE actorSurName NOT IN (
        SELECT actorSurName FROM Series
    )
    ORDER BY actorSurName
''')
df = pd.DataFrame(cursor.fetchall(), columns=['First Name','Surname'])
print(f'Actors in Movies but NOT in Series: {len(df)}')
print(df.head(15).to_string(index=False))

#### Q6c – Additional Query 1: Movies with above-average ratings (WHERE)

In [ ]:
cursor.execute('''
    SELECT movieID, movieName, genre, rating
    FROM Movies
    WHERE rating > (SELECT AVG(rating) FROM Movies)
    ORDER BY rating DESC
    LIMIT 10
''')
df = pd.DataFrame(cursor.fetchall(), columns=['movieID','movieName','genre','rating'])
print('Top 10 movies with above-average ratings:')
print(df.to_string(index=False))

#### Q6d – Additional Query 2: Total views per genre (GROUP BY)

In [ ]:
cursor.execute('''
    SELECT genre, COUNT(*) AS total_views
    FROM UserMovieActivity
    GROUP BY genre
    ORDER BY total_views DESC
''')
df = pd.DataFrame(cursor.fetchall(), columns=['Genre','Total Views'])
print('Movie views by genre:')
print(df.to_string(index=False))

#### Q6e – Additional Query 3: Genres with average series rating > 3 (GROUP BY + HAVING)

In [ ]:
cursor.execute('''
    SELECT genre,
           COUNT(*)        AS series_count,
           AVG(rating)     AS avg_rating
    FROM Series
    GROUP BY genre
    HAVING AVG(rating) > 3
    ORDER BY avg_rating DESC
''')
df = pd.DataFrame(cursor.fetchall(), columns=['Genre','Series Count','Avg Rating'])
print('Genres where average series rating > 3:')
print(df.to_string(index=False))

---
## Part 2: Big Data Concepts (20 Marks)

### 2.1 What is Big Data? The 3 Vs

**Big Data** refers to datasets so large, fast-moving, or complex that traditional data processing tools cannot handle them effectively. IBM (2020) describes Big Data as characterised by the **3 Vs**:

| V | Definition | Ireflix Example |
|---|---|---|
| **Volume** | Massive scale of data generated | Billions of viewing events per day across 50M users |
| **Velocity** | Speed at which data is generated and processed | Real-time recommendation updates per play |
| **Variety** | Different types and formats of data | User profiles, video metadata, ratings, logs, subtitles |

Some frameworks extend this to 5 Vs, adding **Veracity** (data quality/trustworthiness) and **Value** (business insight derived).

### 2.2 Two Big Data Challenges for Ireflix at 50 Million Users

1. **Real-time recommendation latency**: As user count grows to 50 million, generating personalised movie/series recommendations within milliseconds becomes exponentially harder. The volume and velocity of clickstream data overwhelm single-node systems, requiring distributed stream processing (e.g. Apache Kafka + Spark Streaming) to maintain sub-second response times.

2. **Storage and retrieval at scale**: Storing viewing histories, ratings, and metadata for 50 million users generates petabytes of data. Traditional relational databases cannot handle this at scale. Ireflix would need distributed storage solutions such as HDFS or cloud-based data lakes (e.g. AWS S3 + Athena) to ensure reliable, cost-efficient storage with fast query capabilities.

### 2.3 Vertical vs Horizontal Scaling

| | Vertical Scaling (Scale Up) | Horizontal Scaling (Scale Out) |
|---|---|---|
| **Definition** | Adding more power (CPU/RAM) to existing servers | Adding more servers to distribute the load |
| **Cost** | Expensive; hardware limits exist | Cost-effective; uses commodity hardware |
| **Fault Tolerance** | Single point of failure | High availability through replication |
| **Scalability Ceiling** | Physical hardware limits | Virtually unlimited |

**Recommended for Ireflix: Horizontal Scaling.**  
Reason: Platforms like Ireflix experience unpredictable, bursty traffic (e.g. a new series launch can spike views overnight). Horizontal scaling allows new nodes to be added on-demand using cloud autoscaling, ensuring availability without downtime — as demonstrated by Netflix's own migration to AWS (Cockcroft, 2010).

### 2.4 Three Real-World Organisations Using Big Data

**1. Netflix**  
- **Tools**: Apache Spark, Apache Kafka, AWS S3, Cassandra, Presto  
- **Strategy**: Netflix ingests over 1 petabyte of data per day from user interactions. Their recommendation engine uses collaborative filtering on Spark clusters. Kafka handles real-time event streaming (play, pause, stop events). This multi-tier architecture separates batch processing from real-time analytics (Izrailevsky & Tsui, 2011).

**2. Amazon**  
- **Tools**: Amazon Redshift, EMR (Elastic MapReduce), DynamoDB, Kinesis  
- **Strategy**: Amazon processes millions of transactions daily across its e-commerce and AWS services. They use DynamoDB for low-latency key-value lookups (product catalogues), Redshift for OLAP queries on sales trends, and Kinesis for real-time clickstream analysis driving dynamic pricing and personalisation.

**3. Spotify**  
- **Tools**: Apache Hadoop, Apache Beam, Google BigQuery, Luigi  
- **Strategy**: Spotify processes over 600GB of new data per day. Their 'Discover Weekly' playlist feature uses collaborative filtering across billions of listening events processed in Hadoop/Beam pipelines. BigQuery is used for ad-hoc analytics by data scientists (Spotify Engineering, 2019).

---
## Part 3: MapReduce Implementation (20 Marks)

### 3.1 Sample Dataset – Viewing Log (15+ records, 4+ genres)

In [ ]:
# Sample Ireflix viewing log: (genre, watchtime_minutes)
viewing_log = [
    ('Action', 95),
    ('Comedy', 88),
    ('Drama', 120),
    ('Horror', 75),
    ('Action', 110),
    ('Sci-Fi', 130),
    ('Comedy', 45),
    ('Drama', 95),
    ('Action', 85),
    ('Horror', 60),
    ('Sci-Fi', 115),
    ('Drama', 140),
    ('Comedy', 70),
    ('Action', 100),
    ('Sci-Fi', 90),
    ('Horror', 80),
    ('Drama', 55),
    ('Comedy', 110),
    ('Action', 75),
    ('Sci-Fi', 145),
]

print(f'Total viewing records: {len(viewing_log)}')
print(f'Genres covered: {len(set(g for g,_ in viewing_log))}')
print('\nViewing Log Sample:')
df_log = pd.DataFrame(viewing_log, columns=['genre','watchtime_minutes'])
print(df_log.to_string(index=False))

### 3.2 Mapper Function

In [ ]:
def mapper(viewing_log):
    """
    Mapper: reads each (genre, watchtime) record and emits (genre, 1)
    key-value pairs to indicate one viewing event per record.
    """
    mapped_output = []
    for genre, watchtime in viewing_log:
        mapped_output.append((genre, 1))
    return mapped_output

mapped = mapper(viewing_log)
print('Mapper Output (genre, 1):')
for item in mapped:
    print(item)

### 3.3 Reducer Function

In [ ]:
def reducer(mapped_output):
    """
    Reducer: groups by genre key and sums the count values
    to produce total view count per genre.
    """
    counts = {}
    for genre, count in mapped_output:
        counts[genre] = counts.get(genre, 0) + count
    # Return sorted by count descending
    return sorted(counts.items(), key=lambda x: x[1], reverse=True)

reduced = reducer(mapped)
print('Reducer Output (genre, total_views):')
for item in reduced:
    print(item)

### 3.4 Main Function – Orchestrates the MapReduce Pipeline

In [ ]:
def main(viewing_log):
    """
    Main function: orchestrates the MapReduce pipeline.
    Calls mapper() then reducer() and prints final results.
    """
    print('=' * 45)
    print('  IREFLIX MapReduce – View Count by Genre')
    print('=' * 45)
    
    # Step 1 – Map phase
    mapped_output = mapper(viewing_log)
    print(f'\n[MAP]    Emitted {len(mapped_output)} key-value pairs')
    
    # Step 2 – Reduce phase
    final_output = reducer(mapped_output)
    print(f'[REDUCE] Aggregated into {len(final_output)} genre buckets')
    
    # Step 3 – Print results
    print('\nFinal Output:')
    print(f'{"Genre":<15} {"View Count":>10}')
    print('-' * 27)
    for genre, count in final_output:
        print(f'{genre:<15} {count:>10}')
    
    most_watched = final_output[0][0]
    print(f'\nMost Watched Genre: {most_watched} ({final_output[0][1]} views)')
    print('=' * 45)
    return final_output

results = main(viewing_log)

### 3.5 Business Insight

The MapReduce result shows that **Action** is Ireflix's most-watched genre with 6 views, which means Ireflix should prioritise acquiring or licensing more Action content to retain and attract its largest user segment.

---
## Visualisation of Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

genres = [r[0] for r in results]
counts = [r[1] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart – MapReduce output
colours = plt.cm.Set2.colors
bars = axes[0].bar(genres, counts, color=colours[:len(genres)], edgecolor='black')
axes[0].set_title('Ireflix – View Count by Genre (MapReduce)', fontweight='bold')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Total Views')
axes[0].bar_label(bars)

# Pie chart – proportion
axes[1].pie(counts, labels=genres, autopct='%1.1f%%', colors=colours[:len(genres)], startangle=140)
axes[1].set_title('Genre View Share (%)', fontweight='bold')

plt.tight_layout()
plt.savefig('Q3_Figure1_MapReduce_Output.png', bbox_inches='tight')
plt.show()
print('Figure saved: Q3_Figure1_MapReduce_Output.png')

In [ ]:
# Database relationship diagram (text-based ERD summary)
print("""
Ireflix Database – Entity Relationship Summary
═══════════════════════════════════════════════

 Users          Actors            Directors
  PK: userName   PK: surName        PK: surName
    |               |                   |
    |          ┌────┴────┐         ┌────┘
    |          ▼         ▼         ▼
    |        Movies ◄──────────► Directors
    |          PK: movieID        FK: directorSurName
    |          FK: actorSurName
    ├──────────────────────────────────────────┐
    ▼                                          ▼
UserMovieActivity                  UserSeriesActivity
 FK: userName                       FK: userName
 FK: movieID                        FK: seriesID
                                          ▲
                                        Series
                                   PK: seriesID
                                   FK: actorSurName
                                   FK: directorSurName
""")

---
## References

- Cockcroft, A. (2010) *Netflix: A culture of learning*. Netflix Technology Blog. Available at: https://netflixtechblog.com (Accessed: 28 April 2026).
- IBM (2020) *What is Big Data?* IBM Analytics. Available at: https://www.ibm.com/analytics/hadoop/big-data-analytics (Accessed: 28 April 2026).
- Izrailevsky, Y. and Tsui, A. (2011) *The Netflix Tech Blog: Completing the Netflix Cloud Migration*. Netflix Technology Blog. Available at: https://netflixtechblog.com (Accessed: 28 April 2026).
- Spotify Engineering (2019) *How Spotify's data infrastructure scales with the growth of music*. Spotify Engineering Blog. Available at: https://engineering.atspotify.com (Accessed: 28 April 2026).
- White, T. (2015) *Hadoop: The Definitive Guide*. 4th edn. Sebastopol: O'Reilly Media.